# 📈 システムトレード入門チュートリアル

このノートブックでは、システムトレードの基礎を対話的に学習します。

## 目次
1. 環境セットアップ
2. データ取得
3. テクニカル指標の計算
4. シンプルな戦略の実装
5. バックテスト
6. 結果の分析

## 1. 環境セットアップ

まず、必要なライブラリをインポートします。

In [ ]:
# 必要なライブラリのインポート
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# グラフの日本語対応と設定
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['axes.grid'] = True

# 警告を非表示
import warnings
warnings.filterwarnings('ignore')

print("✅ ライブラリのインポート完了！")

## 2. データ取得

Yahoo Finance APIを使用して株価データを取得します。

In [ ]:
# 銘柄と期間を設定
TICKER = "AAPL"  # Apple Inc.
PERIOD = "2y"    # 過去2年間

# データ取得
print(f"📊 {TICKER} のデータを取得中...")
df = yf.Ticker(TICKER).history(period=PERIOD)

print(f"✅ {len(df)} 日分のデータを取得しました")
print(f"📅 期間: {df.index[0].strftime('%Y-%m-%d')} 〜 {df.index[-1].strftime('%Y-%m-%d')}")
print("\n最新5日間のデータ:")
df.tail()

In [ ]:
# 株価チャートを表示
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 終値
axes[0].plot(df.index, df['Close'], color='blue', linewidth=1)
axes[0].set_ylabel('終値 ($)')
axes[0].set_title(f'{TICKER} 株価チャート')

# 出来高
axes[1].bar(df.index, df['Volume'], color='steelblue', alpha=0.7)
axes[1].set_ylabel('出来高')
axes[1].set_xlabel('日付')

plt.tight_layout()
plt.show()

## 3. テクニカル指標の計算

主要なテクニカル指標を計算して可視化します。

In [ ]:
# 移動平均線の計算
df['SMA_20'] = df['Close'].rolling(window=20).mean()  # 20日移動平均
df['SMA_50'] = df['Close'].rolling(window=50).mean()  # 50日移動平均
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()  # 12日指数移動平均

print("✅ 移動平均線を計算しました")
df[['Close', 'SMA_20', 'SMA_50', 'EMA_12']].tail()

In [ ]:
# RSI（相対力指数）の計算
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI'] = calculate_rsi(df['Close'])

print("✅ RSIを計算しました")
print(f"最新のRSI: {df['RSI'].iloc[-1]:.2f}")

In [ ]:
# MACDの計算
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema_12 - ema_26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

print("✅ MACDを計算しました")

In [ ]:
# ボリンジャーバンドの計算
window = 20
df['BB_Middle'] = df['Close'].rolling(window=window).mean()
df['BB_Std'] = df['Close'].rolling(window=window).std()
df['BB_Upper'] = df['BB_Middle'] + 2 * df['BB_Std']
df['BB_Lower'] = df['BB_Middle'] - 2 * df['BB_Std']

print("✅ ボリンジャーバンドを計算しました")

In [ ]:
# テクニカル指標を可視化
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# 1. 株価と移動平均線、ボリンジャーバンド
ax1 = axes[0]
ax1.plot(df.index, df['Close'], label='終値', color='black', linewidth=1)
ax1.plot(df.index, df['SMA_20'], label='SMA(20)', color='blue', alpha=0.7)
ax1.plot(df.index, df['SMA_50'], label='SMA(50)', color='red', alpha=0.7)
ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'], alpha=0.2, color='gray')
ax1.set_ylabel('価格')
ax1.set_title(f'{TICKER} テクニカル分析')
ax1.legend(loc='upper left')

# 2. RSI
ax2 = axes[1]
ax2.plot(df.index, df['RSI'], color='purple', linewidth=1)
ax2.axhline(y=70, color='red', linestyle='--', alpha=0.5)
ax2.axhline(y=30, color='green', linestyle='--', alpha=0.5)
ax2.fill_between(df.index, df['RSI'], 70, where=(df['RSI'] >= 70), alpha=0.3, color='red')
ax2.fill_between(df.index, df['RSI'], 30, where=(df['RSI'] <= 30), alpha=0.3, color='green')
ax2.set_ylabel('RSI')
ax2.set_ylim(0, 100)

# 3. MACD
ax3 = axes[2]
ax3.plot(df.index, df['MACD'], label='MACD', color='blue', linewidth=1)
ax3.plot(df.index, df['MACD_Signal'], label='Signal', color='red', linewidth=1)
colors = ['green' if val >= 0 else 'red' for val in df['MACD_Hist']]
ax3.bar(df.index, df['MACD_Hist'], color=colors, alpha=0.5)
ax3.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax3.set_ylabel('MACD')
ax3.legend(loc='upper left')

# 4. 出来高
ax4 = axes[3]
ax4.bar(df.index, df['Volume'], color='steelblue', alpha=0.7)
ax4.set_ylabel('出来高')
ax4.set_xlabel('日付')

plt.tight_layout()
plt.show()

## 4. シンプルな戦略の実装

移動平均クロス戦略を実装します。

**ルール:**
- 短期移動平均が長期移動平均を上抜けたら **買い**（ゴールデンクロス）
- 短期移動平均が長期移動平均を下抜けたら **売り**（デッドクロス）

In [ ]:
# 移動平均クロス戦略のシグナル生成
SHORT_WINDOW = 20
LONG_WINDOW = 50

# シグナル列を作成
df['Signal'] = 0

for i in range(1, len(df)):
    # ゴールデンクロス（短期が長期を上抜け）→ 買い
    if (df['SMA_20'].iloc[i] > df['SMA_50'].iloc[i] and 
        df['SMA_20'].iloc[i-1] <= df['SMA_50'].iloc[i-1]):
        df.loc[df.index[i], 'Signal'] = 1
    
    # デッドクロス（短期が長期を下抜け）→ 売り
    elif (df['SMA_20'].iloc[i] < df['SMA_50'].iloc[i] and 
          df['SMA_20'].iloc[i-1] >= df['SMA_50'].iloc[i-1]):
        df.loc[df.index[i], 'Signal'] = -1

buy_signals = df[df['Signal'] == 1]
sell_signals = df[df['Signal'] == -1]

print(f"✅ シグナル生成完了")
print(f"📈 買いシグナル: {len(buy_signals)} 回")
print(f"📉 売りシグナル: {len(sell_signals)} 回")

In [ ]:
# シグナルを可視化
plt.figure(figsize=(14, 7))

plt.plot(df.index, df['Close'], label='終値', color='black', linewidth=1)
plt.plot(df.index, df['SMA_20'], label=f'SMA({SHORT_WINDOW})', color='blue', alpha=0.7)
plt.plot(df.index, df['SMA_50'], label=f'SMA({LONG_WINDOW})', color='red', alpha=0.7)

# 買いシグナル
plt.scatter(buy_signals.index, buy_signals['Close'], 
            marker='^', color='green', s=150, label='買い', zorder=5)

# 売りシグナル
plt.scatter(sell_signals.index, sell_signals['Close'],
            marker='v', color='red', s=150, label='売り', zorder=5)

plt.title(f'{TICKER} - 移動平均クロス戦略')
plt.xlabel('日付')
plt.ylabel('価格 ($)')
plt.legend(loc='upper left')
plt.show()

## 5. バックテスト

戦略のパフォーマンスを検証します。

In [ ]:
# シンプルなバックテスト
INITIAL_CAPITAL = 1000000  # 初期資金: 100万円
COMMISSION = 0.001  # 手数料: 0.1%

capital = INITIAL_CAPITAL
position = 0  # 保有株数
trades = []   # 取引履歴
equity_curve = []  # 資産推移

entry_price = 0

for i in range(len(df)):
    date = df.index[i]
    price = df['Close'].iloc[i]
    signal = df['Signal'].iloc[i]
    
    # 現在の資産価値を記録
    current_equity = capital + position * price
    equity_curve.append({'date': date, 'equity': current_equity})
    
    # 買いシグナル & ポジションなし
    if signal == 1 and position == 0:
        # 手数料を考慮して購入
        shares = (capital * (1 - COMMISSION)) / price
        position = shares
        capital = 0
        entry_price = price
        trades.append({
            'type': 'BUY',
            'date': date,
            'price': price,
            'shares': shares
        })
    
    # 売りシグナル & ポジションあり
    elif signal == -1 and position > 0:
        # 売却
        proceeds = position * price * (1 - COMMISSION)
        profit = proceeds - (position * entry_price)
        capital = proceeds
        trades.append({
            'type': 'SELL',
            'date': date,
            'price': price,
            'shares': position,
            'profit': profit
        })
        position = 0

# 最終日にポジションが残っている場合
if position > 0:
    final_price = df['Close'].iloc[-1]
    capital = position * final_price * (1 - COMMISSION)
    position = 0

equity_df = pd.DataFrame(equity_curve).set_index('date')

print(f"✅ バックテスト完了")
print(f"\n💰 初期資金: ¥{INITIAL_CAPITAL:,.0f}")
print(f"💰 最終資金: ¥{capital:,.0f}")
print(f"📈 リターン: {((capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100):+.2f}%")

In [ ]:
# 取引履歴を表示
trades_df = pd.DataFrame(trades)
print("📜 取引履歴:")
trades_df

In [ ]:
# パフォーマンス指標を計算

# 総リターン
total_return = (capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100

# 取引統計
sell_trades = trades_df[trades_df['type'] == 'SELL']
if 'profit' in sell_trades.columns:
    winning_trades = sell_trades[sell_trades['profit'] > 0]
    losing_trades = sell_trades[sell_trades['profit'] <= 0]
    win_rate = len(winning_trades) / len(sell_trades) * 100 if len(sell_trades) > 0 else 0
    
    avg_win = winning_trades['profit'].mean() if len(winning_trades) > 0 else 0
    avg_loss = losing_trades['profit'].mean() if len(losing_trades) > 0 else 0
    
    total_profit = winning_trades['profit'].sum() if len(winning_trades) > 0 else 0
    total_loss = abs(losing_trades['profit'].sum()) if len(losing_trades) > 0 else 1
    profit_factor = total_profit / total_loss if total_loss > 0 else float('inf')
else:
    win_rate = 0
    avg_win = 0
    avg_loss = 0
    profit_factor = 0

# 最大ドローダウン
running_max = equity_df['equity'].cummax()
drawdown = (equity_df['equity'] - running_max) / running_max * 100
max_drawdown = drawdown.min()

# シャープレシオ
daily_returns = equity_df['equity'].pct_change().dropna()
sharpe_ratio = daily_returns.mean() / daily_returns.std() * np.sqrt(252) if daily_returns.std() > 0 else 0

# Buy & Hold リターン
buy_hold_return = (df['Close'].iloc[-1] - df['Close'].iloc[0]) / df['Close'].iloc[0] * 100

print("=" * 50)
print("📊 パフォーマンス指標")
print("=" * 50)
print(f"\n💰 リターン")
print(f"  戦略リターン: {total_return:+.2f}%")
print(f"  Buy & Hold: {buy_hold_return:+.2f}%")
print(f"\n📈 取引統計")
print(f"  取引回数: {len(sell_trades)}")
print(f"  勝率: {win_rate:.1f}%")
print(f"  プロフィットファクター: {profit_factor:.2f}")
print(f"\n⚠️ リスク指標")
print(f"  最大ドローダウン: {max_drawdown:.2f}%")
print(f"  シャープレシオ: {sharpe_ratio:.2f}")

In [ ]:
# 資産推移とドローダウンを可視化
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 資産推移
ax1 = axes[0]
ax1.plot(equity_df.index, equity_df['equity'], color='blue', linewidth=1)
ax1.axhline(y=INITIAL_CAPITAL, color='gray', linestyle='--', alpha=0.5, label='初期資金')
ax1.set_ylabel('資産額 (¥)')
ax1.set_title(f'{TICKER} - 移動平均クロス戦略 資産推移')
ax1.legend(loc='upper left')

# ドローダウン
ax2 = axes[1]
ax2.fill_between(equity_df.index, drawdown, 0, alpha=0.5, color='red')
ax2.set_ylabel('ドローダウン (%)')
ax2.set_xlabel('日付')

plt.tight_layout()
plt.show()

## 6. 結果の分析

バックテスト結果を分析し、戦略の改善点を考えます。

In [ ]:
# Buy & Hold と戦略のパフォーマンスを比較
# Buy & Hold の資産推移を計算
initial_shares = INITIAL_CAPITAL / df['Close'].iloc[0]
buy_hold_equity = initial_shares * df['Close']

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(equity_df.index, equity_df['equity'], label='移動平均クロス戦略', color='blue', linewidth=1.5)
ax.plot(df.index, buy_hold_equity, label='Buy & Hold', color='gray', linewidth=1.5, alpha=0.7)
ax.axhline(y=INITIAL_CAPITAL, color='red', linestyle='--', alpha=0.5, label='初期資金')

ax.set_ylabel('資産額 (¥)')
ax.set_xlabel('日付')
ax.set_title('戦略 vs Buy & Hold パフォーマンス比較')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 🎯 チャレンジ課題

以下の課題に挑戦してみましょう：

1. **パラメータの変更**
   - 短期・長期移動平均の期間を変えてみる
   - どのパラメータが最も良いパフォーマンスを出すか？

2. **別の戦略を実装**
   - RSI戦略（RSI < 30 で買い、RSI > 70 で売り）
   - MACD戦略（MACDがシグナルを上抜けで買い）

3. **リスク管理の追加**
   - ストップロス（損切り）を追加
   - トレーリングストップを実装

4. **別の銘柄でテスト**
   - 日本株（例: 7203.T トヨタ）
   - 他の米国株（例: GOOGL, MSFT）

In [ ]:
# チャレンジ用のコードセル
# ここに自分のコードを書いてみましょう！



## 📚 次のステップ

このチュートリアルを終えたら、以下に進んでみましょう：

1. `src/04_strategies.py` - より高度な戦略の実装
2. 複数銘柄でのポートフォリオ戦略
3. 機械学習を使った予測モデル

---

⚠️ **注意**: このノートブックは学習目的です。実際の取引を行う前に、十分なテストと理解が必要です。投資には常にリスクが伴います。